# Лунная База v2.4: решение команды

Выберите задание «Свободный заезд», нажмите «Новый прогон» и выполняйте ячейки сверху вниз.

Документы сценария:

- «Задачи робота» — порядок заезда и условия каждой задачи;
- «Задачи кибериммунности» — что делает модуль безопасности, протоколы и журнал МБ;
- «Python API участника» — методы клиента, готовые движения, события и ошибки;
- «Обозначения и полигон» — клетки, зоны и система координат;
- «Механика робота и полигона» — движение, захват, шлагбаумы и индикация.

In [1]:
# Подгтовка заезда

from quasar_client import QuasarClient;
from math import pi;
# для форматированного (более читаемого) вывода словарей
# убрать из финального кода
#from pprint import pp;
from time import sleep;


# запишем некоторые координаты константами
# (например, положение стержней)
# их нет в state, и они, видимо, могут отличаться на настоящем
# полигоне, поэтому можно сделать так, чтобы потом не искать в коде
# положения стержней, безопасное расстояние для робота при подъезде к катушке, может, что-то ещё

# смещение по X от катушки перед тем, как её брать
# для сравнения: длина корпуса = 0.259
SPOOL_START_OFFSET_X = 0.17;

# x-координата для третьего стержня
THIRD_ROD_X = 2.36;

# начальное Y-положение перед настройкой стержней
# (чтобы не задеть стержень при дальнейшем движении)
ROD_START_Y = 2.85;

# промежуток между стержнями
ROD_DX = 0.19;


ID = "robot.1";

client = QuasarClient.from_environment(timeout=5.0,queue_size=100);

'''
НАСТРОЙКА ЗАДАЧ КИБЕРБЕЗОПАСНОСТИ
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№

№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№

№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№№
'''
# пока что для тестирования всё выключено
# @&$!*#
client.configure_incidents([]);

st = client.get_state # зачем писать больше, если можно писать меньше?

client.wait_for_simulation_start();



robot = client.get_robot(ID);
rod_targets = [rod['targetOffset'] for rod in client.get_rod_targets()['rods']];


def move_to(cell_col, cell_row, x_offset=0.0, y_offset=0.0, rot=None):
    cell_idx = cell_col + (cell_row - 1) * 9;
    x, y = QuasarClient.cell_center(cell_idx);
    x += x_offset;
    y += y_offset;
    client.move_to(ID, x, y, rot, speed=0.3);


# тоже использует координаты клетки (пусть cells будет список кортежй)
def move_path(cells):
    for x, y in cells:
        move_to(x, y); # всё равно ведь тоже самое? только списком   


def gate_angle(state, gate_id):
    return next(g['angle'] for g in state['gates'] if g['id'] == gate_id);

    
def wait_for_gate(gate_id):
    client.open_gate(gate_id);
    client.wait_until(lambda s: gate_angle(s, gate_id) > 1.4, timeout=5);
        

# new idea
# в общем, тот же client.move_to()
# но без замедления и ожидания остановки
# то есть после достижения (в каких-то пределах) целевой точки
# робот незамедлительно приступает к выполнению следующей команды
# не знаю точно, но, возможно, на этом можно будет несколько секунд сэкономить?
# библиотеки поменяли, поэтому этот код тоже не работает
# def move_to_quick(
#         x: float,
#         y: float,
#         rotation: float | None = None,
#         speed: float = 0.3, # hey!!!! вот почему он медленно двигался
#         tolerance: float = 0.01,
#         angle_tolerance: float = 0.03,
#         timeout: float = 30.0,
# ) -> dict:
#     from quasar_client import _clamp, _Progress, _wrap_angle, QuasarTimeoutError, _CONTROL_PERIOD;
#     from math import hypot;
#     from time import monotonic;
#     # это всё проверки, можно их убрать пока что
#     #robot_id = _robot_id(robot_id)
#     robot_id = ID;
#     #target_x, target_y = _inside_arena("x", x), _inside_arena("y", y)
#     target_x, target_y = x, y;
#     #target_rotation = None if rotation is None else _finite("rotation", rotation)
#     target_rotation = rotation;
#     #speed = _positive("speed", speed)
#     #tolerance = _positive("tolerance", tolerance)
#     #angle_tolerance = _positive("angle_tolerance", angle_tolerance)
#     #timeout = _positive("timeout", timeout)
#     deadline = monotonic() + timeout
#     next_cycle = monotonic()
#     progress = _Progress()
#     try:
#         while True:
#             state = st()
#             # px, py - положение робота в данный момент времени
#             px, py, heading = client._pose(state, ID)
#             progress.observe((px, py, heading))
#             # смещение робота от цели
#             dx, dy = target_x - px, target_y - py
#             # и расстояние до цели
#             distance = hypot(dx, dy)
#             angle_error = 0.0 if target_rotation is None else _wrap_angle(target_rotation - heading)
#             # доехал
#             if distance <= tolerance and abs(angle_error) <= angle_tolerance:
#                 client.stop(ID)
#                 return;
#                 return client._robot(client.get_state(), robot_id)
#             # робот не доехал (не успел)
#             if monotonic() >= deadline:
#                 raise QuasarTimeoutError(
#                     f"move_to: за {timeout:g} с робот не доехал до цели, осталось {distance:.3f} м"
#                     + (f" и {abs(angle_error):.3f} рад до нужного курса" if abs(angle_error) > angle_tolerance else "")
#                     + f". Вероятная причина: {progress.reason(client._robot(state, robot_id))}"
#                 )
#             if distance <= tolerance:
#                 # vx, vy - скорости
#                 vx = vy = 0.0
#             else:
#                 # теперь замедляется совсем чуть-чуть при приближении к цели
#                 cruise = min(speed, max(0.2, 2.0 * distance))
#                 vx, vy = cruise * dx / distance, cruise * dy / distance
#             angular = 0.0
#             if target_rotation is not None and abs(angle_error) > angle_tolerance * 0.5:
#                 angular = _clamp(2.5 * angle_error, 1.5)
#             progress.command(vx, vy, angular, client.set_velocity(ID, vx, vy, angular, 'world', heading=heading))
#             next_cycle += _CONTROL_PERIOD
#             sleep(max(0.0, next_cycle - monotonic()))
#     except BaseException:
#         client._try_stop(robot_id)
#         raise



# CybP_04: None — готовые движения едут по глобальной навигации
# CybP_02: цикл обновлений Диспетчера — здесь, до первого движения робота.
# CybP_03: монитор дозиметра — фоновый поток, опрос 5–10 раз в секунду.



In [2]:

# киберимунность — единый блок CybP_02…CybP_05

import base64, binascii, hashlib, json, math, struct, threading, time
from cryptography.hazmat.primitives.asymmetric import x25519, ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidSignature, InvalidTag

_MB = {
    "lock": threading.RLock(), "last_motor_seq": None,
    "last_feedback_seq": None, "last_ticks": None, "last_feedback_t": None,
    "pose": None, "yaw_offset": None, "local_active": False,
    "last_dose_high": False, "dose_cell": None, "dose_high_since": None,
    "last_motor_log": 0.0, "motor_block_count": 0, "motor_reasons": {},
    "feedback_bad": 0, "last_feedback_log": 0.0, "feedback_reasons": {},
    "last_feedback_raw": None, "last_feedback_seen_seq": None,
    "dose_stop": threading.Event(), "dose_thread": None,
}

def _decode(v):
    if isinstance(v, bytes):
        return v
    if isinstance(v, str):
        if v.startswith("-----BEGIN "):
            return v.encode("ascii")
        return base64.b64decode(v, validate=True)
    raise TypeError("Ожидались bytes или строка")

def _ver(v):
    try: return tuple(int(x) for x in str(v).split("."))
    except (ValueError, TypeError): return (0,)

def _log(level, event, message, **details):
    try:
        client.journal(level, event, str(message)[:280], **details)
    except Exception as exc:
        try: client.warning(f"МБ: запись {event} не удалась ({type(exc).__name__})")
        except Exception: pass

def _cell_number(x, y):
    if x is None or y is None or not (0 <= float(x) < 3.6 and 0 <= float(y) < 3.6):
        return None
    col, row = int(float(x) / 0.4), int(float(y) / 0.4)
    return min(8, row) * 9 + min(8, col) + 1


# ---------------- CybP_02: один цикл проверки обновлений ----------------
def install_verified_firmware(max_sessions=8):
    identity = client.get_update_identity()
    required = {"grip-controller", "motor-driver", "sensor-hub"}
    current = {x["module"]: x.get("version", "0.0.0") for x in identity.get("firmware", [])}
    states = {x["module"]: x.get("state") for x in identity.get("firmware", [])}

    if not identity.get("updatesAvailable", False):
        if required.issubset(current) and all(states.get(m) not in {"installing", "failed"} for m in required):
            return
        raise RuntimeError("CybP_02: обязательные прошивки не подтверждены; движение запрещено")

    device_private = x25519.X25519PrivateKey.from_private_bytes(_decode(identity["devicePrivateKey"]))
    device_pub_hex = device_private.public_key().public_bytes(
        serialization.Encoding.Raw, serialization.PublicFormat.Raw
    ).hex()
    pub_data = _decode(identity["dispatcherPublicKey"])
    if not pub_data.startswith(b"-----BEGIN "):
        raise RuntimeError("CybP_02: открытый ключ Диспетчера не в формате PEM")
    dispatcher_pub = serialization.load_pem_public_key(pub_data)
    installed_this_run = set()

    for _ in range(max_sessions):
        session = client.open_update_session()
        sid = str(session["sessionId"])
        dispatcher_raw = _decode(session["dispatcherKey"])
        signed_session = f"{sid}|{dispatcher_raw.hex()}|{device_pub_hex}".encode()
        try:
            dispatcher_pub.verify(_decode(session["signature"]), signed_session, ec.ECDSA(hashes.SHA256()))
        except (InvalidSignature, ValueError, TypeError):
            client.reject_update(sid, "Подпись сеанса не подтверждена")
            _log("error", "CybP_02", "Отклонён сеанс с недействительной подписью",
                 sessionId=sid, reason="invalid-session-signature")
            continue

        shared = device_private.exchange(x25519.X25519PublicKey.from_public_bytes(dispatcher_raw))
        key = HKDF(algorithm=hashes.SHA256(), length=32, salt=sid.encode(),
                   info=b"moonbase-update-channel-v1").derive(shared)
        aes = AESGCM(key)
        try:
            env = client.get_update_manifest(sid)
            manifest = json.loads(aes.decrypt(_decode(env["nonce"]), _decode(env["ciphertext"]),
                                               f"{sid}|manifest".encode()).decode())
        except Exception as exc:
            _log("error", "CybP_02", "Манифест не прошёл проверку аутентичности",
                 sessionId=sid, reason=type(exc).__name__)
            raise RuntimeError("CybP_02: манифест не подтверждён; движение запрещено") from exc

        for pkg in manifest.get("packages", []):
            pid, module = str(pkg.get("id", "unknown")), str(pkg.get("module", ""))
            reason, image = None, None
            try:
                env = client.fetch_update(sid, pid)
                image = aes.decrypt(_decode(env["nonce"]), _decode(env["ciphertext"]), f"{sid}|{pid}".encode())
                digest = hashlib.sha256(image).hexdigest()
                if digest.lower() != str(pkg["sha256"]).lower():
                    reason = "sha256-mismatch"
                elif pkg.get("target") != identity.get("hardware"):
                    reason = "hardware-target-mismatch"
                elif module not in required:
                    reason = "unexpected-module"
                elif _ver(pkg.get("version", "0")) <= _ver(current.get(module, "0")):
                    reason = "version-not-newer"
                else:
                    signed_pkg = (f'{pkg["id"]}|{pkg["module"]}|{pkg["version"]}|'
                                  f'{pkg["target"]}|{pkg["sha256"]}').encode()
                    dispatcher_pub.verify(_decode(pkg["signature"]), signed_pkg, ec.ECDSA(hashes.SHA256()))
            except (InvalidSignature, InvalidTag, KeyError, ValueError, TypeError):
                reason = "signature-or-format-invalid"
            except Exception as exc:
                reason = "package-verification-error-" + type(exc).__name__

            if reason:
                client.reject_update(sid, reason, pid)
                _log("error", "CybP_02", "Пакет отклонён до установки", packageId=pid, reason=reason)
                continue

            client.install_update(sid, pid, digest)
            refreshed = client.wait_for_firmware(timeout=10)
            current = {x["module"]: x.get("version", "0.0.0") for x in refreshed.get("firmware", [])}
            states = {x["module"]: x.get("state") for x in refreshed.get("firmware", [])}
            if current.get(module) != pkg.get("version") or states.get(module) == "failed":
                raise RuntimeError(f"CybP_02: установка {module} не подтверждена")
            installed_this_run.add(module)
            _log("info", "CybP_02", "Прошивка проверена и установлена",
                 packageId=pid, module=module, version=pkg["version"])

        final = client.get_update_identity()
        current = {x["module"]: x.get("version", "0.0.0") for x in final.get("firmware", [])}
        states = {x["module"]: x.get("state") for x in final.get("firmware", [])}
        if required.issubset(current) and all(states.get(m) not in {"installing", "failed"} for m in required):
            if not final.get("updatesAvailable", False) or required.issubset(installed_this_run):
                return

    raise RuntimeError("CybP_02: не подтверждены все обязательные прошивки; движение запрещено")

install_verified_firmware()


# ---------------- CybP_05: фильтр команд двигателей ----------------
def motor_filter(frame, command):
    """CybP_05: блокирует кадр до передачи драйверу, если он повреждён."""
    reason = None
    seq_is_trustworthy = False
    seq = None

    try:
        if not isinstance(frame, (bytes, bytearray)) or len(frame) != 16:
            reason = "bad-length"
        elif frame[0] != 0xA5 or frame[1] != 0x01:
            reason = "bad-header"
        elif struct.unpack_from("<H", frame, 14)[0] != binascii.crc_hqx(frame[:14], 0xFFFF):
            # при неверной CRC нельзя доверять даже полю seq
            reason = "bad-crc"
        else:
            seq = frame[2]
            seq_is_trustworthy = True

            # поля флагов и резерва по протоколу должны быть нулевыми
            if frame[3] != 0:
                reason = "bad-flags"
            elif frame[12] != 0 or frame[13] != 0:
                reason = "nonzero-reserved-bytes"

            prev = _MB["last_motor_seq"]
            if reason is None and prev is not None:
                advance = (seq - prev) & 0xFF
                if not (0 < advance < 128):
                    reason = "replayed-sequence"

            rpm = [v / 10.0 for v in struct.unpack_from("<4h", frame, 4)]
            if reason is None and any(abs(v) > QuasarClient.RATED_RPM + 1e-9 for v in rpm):
                reason = "wheel-speed-over-limit"

            if reason is None:
                vx = float(command["x"])
                vy = float(command["y"])
                w = float(command["angular"])
                r, L = QuasarClient.WHEEL_RADIUS, QuasarClient.YAW_ARM
                rad_s = [
                    (vx - vy - L*w) / r,  # FL
                    (vx + vy + L*w) / r,  # FR
                    (vx + vy - L*w) / r,  # RL
                    (vx - vy + L*w) / r,  # RR
                ]
                expected = [v * 60.0 / (2.0 * math.pi) for v in rad_s]
                # кадр хранит RPM с точностью 0.1
                # запас 0.16 RPM покрывает округлениe
                # но не маскирует подмену
                if any(abs(actual - target) > 0.16 for actual, target in zip(rpm, expected)):
                    reason = "wheel-command-mismatch"

    except Exception:
        reason = "frame-validation-error"

    # seq увеличивается на каждом сформированном кадре, в том числе отклонённом фильтром
    # запоминаем его, если заголовок и CRC достоверны,
    # иначе повторный повреждённый кадр можно было бы принять позднее
    if seq_is_trustworthy:
        prev = _MB["last_motor_seq"]
        if prev is None or 0 < ((seq - prev) & 0xFF) < 128:
            _MB["last_motor_seq"] = seq

    if reason:
        now = time.monotonic()
        _MB["motor_block_count"] += 1
        _MB["motor_reasons"][reason] = _MB["motor_reasons"].get(reason, 0) + 1
        if now - _MB["last_motor_log"] >= 1.0:
            summary = ",".join(f"{k}:{v}" for k, v in sorted(_MB["motor_reasons"].items()))
            _log(
                "warning", "CybP_05",
                f"Заблокированы недостоверные кадры; всего={_MB['motor_block_count']}; {summary}"
            )
            _MB["last_motor_log"], _MB["motor_reasons"] = now, {}
        return reason
    return None


# ---------------- Обратная связь и CybP_04: локальная поза ----------------
def _feedback_reject(reason):
    """Сводно журналирует недостоверную обратную связь, не чаще раза в секунду."""
    now = time.monotonic()
    _MB["feedback_bad"] += 1
    reasons = _MB["feedback_reasons"]
    reasons[reason] = reasons.get(reason, 0) + 1
    if now - _MB["last_feedback_log"] >= 1.0:
        summary = ",".join(f"{k}:{v}" for k, v in sorted(reasons.items()))
        _log(
            "warning", "CybP_05",
            f"Отброшена недостоверная обратная связь; всего={_MB['feedback_bad']}; {summary}"
        )
        _MB["last_feedback_log"] = now
        _MB["feedback_reasons"] = {}


def _parse_feedback(robot):
    """Проверяет кадр обратной связи; одинаковый закэшированный кадр не считается атакой."""
    raw = (robot.get("sensors") or {}).get("motorFeedback")
    if not raw:
        return None
    try:
        frame = bytes.fromhex(raw) if isinstance(raw, str) else bytes(raw)
    except (TypeError, ValueError):
        _feedback_reject("bad-encoding")
        return None

    if len(frame) != 24:
        _feedback_reject("bad-length")
        return None
    if frame[:2] != b"\xA5\x02":
        _feedback_reject("bad-header")
        return None
    if struct.unpack_from("<H", frame, 22)[0] != binascii.crc_hqx(frame[:22], 0xFFFF):
        _feedback_reject("bad-crc")
        return None
    if frame[3] != 0:
        _feedback_reject("bad-status")
        return None
    temp = struct.unpack_from("<h", frame, 20)[0] / 10.0
    if not -40 <= temp <= 120:
        _feedback_reject("temperature-out-of-range")
        return None

    seq = frame[2]
    previous_raw = _MB["last_feedback_raw"]
    previous_seq = _MB["last_feedback_seen_seq"]

    if previous_raw == frame:
        return None

    if previous_seq is not None:
        advance = (seq - previous_seq) & 0xFF
        if not (0 < advance < 128):
            _feedback_reject("replayed-sequence")
            return None

    _MB["last_feedback_raw"] = frame
    _MB["last_feedback_seen_seq"] = seq
    ticks = struct.unpack_from("<4i", frame, 4)
    return seq, ticks

def pose_provider(robot):
    with _MB["lock"]:
        gx = float(robot.get("x", 0.2))
        gy = float(robot.get("y", 1.4))
        gr = float(robot.get("rotation", 0.0))
        sensors = robot.get("sensors") or {}
        gyro = sensors.get("gyro") or {}
        yaw = gyro.get("yaw")
        dose = (sensors.get("dosimeter") or {}).get("doseRate")
        now = time.monotonic()
        fb = _parse_feedback(robot)

        if _MB["pose"] is None:
            _MB["pose"] = [gx, gy, gr]
            if yaw is not None:
                _MB["yaw_offset"] = gr - float(yaw)
            _MB["last_feedback_t"] = now
            if fb:
                seq, ticks = fb
                _MB["last_feedback_seq"] = seq
                _MB["last_ticks"] = ticks
        elif fb:
            seq, ticks = fb
            prev_ticks = _MB["last_ticks"]
            prev_t = _MB["last_feedback_t"]
            accepted_ticks = False

            if prev_ticks is None:
                accepted_ticks = True
            else:
                dt = max(0.001, now - (prev_t if prev_t is not None else now))
                delta = [ticks[i] - prev_ticks[i] for i in range(4)]
                # предел физически достижимого перемещения при номинальных оборотах
                max_ticks = (
                    QuasarClient.RATED_RPM / 60.0
                    * QuasarClient.TICKS_PER_REVOLUTION * dt + 4
                )
                if max(abs(v) for v in delta) <= max_ticks:
                    r = QuasarClient.WHEEL_RADIUS
                    L = QuasarClient.YAW_ARM
                    d = [v * 2.0 * math.pi / QuasarClient.TICKS_PER_REVOLUTION for v in delta]
                    dx_body = r / 4.0 * sum(d)
                    dy_body = r / 4.0 * (-d[0] + d[1] + d[2] - d[3])

                    theta_before = _MB["pose"][2]
                    if yaw is not None and _MB["yaw_offset"] is not None:
                        theta_now = (
                            float(yaw) + _MB["yaw_offset"] + math.pi
                        ) % (2.0 * math.pi) - math.pi
                        dtheta = (theta_now - theta_before + math.pi) % (2.0 * math.pi) - math.pi
                        theta_mid = theta_before + dtheta / 2.0
                        _MB["pose"][0] += (
                            dx_body * math.cos(theta_mid) - dy_body * math.sin(theta_mid)
                        )
                        _MB["pose"][1] += (
                            dx_body * math.sin(theta_mid) + dy_body * math.cos(theta_mid)
                        )
                        _MB["pose"][2] = theta_now
                    else:
                        _MB["pose"][0] += (
                            dx_body * math.cos(theta_before) - dy_body * math.sin(theta_before)
                        )
                        _MB["pose"][1] += (
                            dx_body * math.sin(theta_before) + dy_body * math.cos(theta_before)
                        )
                        da = r / (4.0 * L) * (-d[0] + d[1] - d[2] + d[3])
                        _MB["pose"][2] = (
                            theta_before + da + math.pi
                        ) % (2.0 * math.pi) - math.pi
                    accepted_ticks = True
                else:
                    _feedback_reject("tick-jump")

            _MB["last_feedback_seq"] = seq
            if accepted_ticks:
                _MB["last_ticks"], _MB["last_feedback_t"] = ticks, now

        # гироскоп корректирует курс и в глобальном, и в локальном режимах
        if yaw is not None:
            if _MB["yaw_offset"] is None:
                _MB["yaw_offset"] = gr - float(yaw)
            _MB["pose"][2] = (
                float(yaw) + _MB["yaw_offset"] + math.pi
            ) % (2.0 * math.pi) - math.pi

        # CybP_03: дозиметрический переход журналируется
        # маршрут обязан выводить робота
        high = dose is not None and float(dose) > 5.0
        if high:
            cell = _cell_number(gx, gy)
            if cell is None:
                cell = _cell_number(_MB["pose"][0], _MB["pose"][1])
            if not _MB["last_dose_high"] or cell != _MB["dose_cell"]:
                _MB["dose_high_since"] = now
                _log(
                    "warning", "CybP_03", "Обнаружена повышенная мощность дозы",
                    cell=cell if cell is not None else -1, dose=round(float(dose), 2)
                )
                _MB["dose_cell"] = cell
        else:
            _MB["dose_high_since"], _MB["dose_cell"] = None, None
        _MB["last_dose_high"] = high

        in_reactor = gy >= 2.35

        if not _MB["local_active"] and not in_reactor:
            _MB["pose"][0], _MB["pose"][1], _MB["pose"][2] = gx, gy, gr
            if yaw is not None:
                _MB["yaw_offset"] = gr - float(yaw)

        lx, ly, lr = _MB["pose"]
        pos_err = math.hypot(gx - lx, gy - ly)
        ang_err = abs((gr - lr + math.pi) % (2.0 * math.pi) - math.pi)
        if not _MB["local_active"] and in_reactor and (pos_err > 0.075 or ang_err > 0.18):
            _MB["local_active"] = True
            cell = _cell_number(gx, gy)
            _log(
                "warning", "CybP_04", "Расхождение навигации: включена локальная оценка",
                cell=cell if cell is not None else -1, error_m=round(pos_err, 3)
            )
            client.report_navigation(ID, "local", lx, ly)
        elif _MB["local_active"] and not in_reactor and pos_err <= 0.07 and ang_err <= 0.15:
            _MB["local_active"] = False
            _log("info", "CybP_04", "Глобальная навигация восстановлена")
            client.report_navigation(ID, "global")
        if _MB["local_active"]:
            return tuple(_MB["pose"])
        return gx, gy, gr


# ---------------- CybP_03: фоновый монитор дозиметра ----------------
# не выдаём движение из второго потока — это может конфликтовать с автопилотом
# если робот не уходит из опасной клетки, пишем событие; основной маршрут должен
# предусматривать выход
# порог/таймеры инцидента не отключаются
def _cybp03_watch():
    warned = False
    while not _MB["dose_stop"].is_set():
        try:
            state = client.get_state()
            if state.get("attempt", {}).get("phase") in {"finished", "stopped"}:
                return
            robot = next((r for r in state.get("robots", []) if r.get("id") == ID), None)
            if robot:
                dose = ((robot.get("sensors") or {}).get("dosimeter") or {}).get("doseRate")
                now = time.monotonic()
                if dose is not None and float(dose) > 5.0:
                    if _MB["dose_high_since"] is None: _MB["dose_high_since"] = now
                    elapsed = now - _MB["dose_high_since"]
                    if elapsed >= 3.0 and not warned:
                        cell = _cell_number(robot.get("x"), robot.get("y"))
                        _log("error", "CybP_03", "Робот задержался в зоне повышенной дозы; требуется выход по маршруту",
                             cell=cell if cell is not None else -1, elapsed_s=round(elapsed, 1))
                        warned = True
                elif dose is not None and float(dose) < 3.0:
                    _MB["dose_high_since"], warned = None, False
        except Exception as exc:
            _log("warning", "CybP_03", "Ошибка чтения дозиметра", reason=type(exc).__name__)
        _MB["dose_stop"].wait(0.2)

def start_cybp03_monitor():
    t = _MB["dose_thread"]
    if t is None or not t.is_alive():
        _MB["dose_stop"].clear()
        _MB["dose_thread"] = threading.Thread(target=_cybp03_watch, name="cybp03-dosimeter", daemon=True)
        _MB["dose_thread"].start()

def stop_cybp03_monitor():
    _MB["dose_stop"].set()

# ...
incidents = st()['incidents']['enabled'];
if len(incidents) != 0:
    print('МБ подключён, угрозы кибериммунности:', ', '.join(incidents), sep=' ');
    client.pose_provider = pose_provider;
    start_cybp03_monitor();
    client.motor_filter = motor_filter;
else:
    print("МБ не подключён, заезд без угроз кибериммунности");
for inc in incidents:
    client.journal('info', inc, 'Инцидент ' + inc + ' активирован');

МБ не подключён, заезд без угроз кибериммунности


## Заезд

Порядок задач, условия и оценка — в документе «Задачи робота».

- Шлагбаум открывается, только когда центр робота стоит в его подходе. Через 15 с после нажатия кнопки реактора
  шлагбаумы блокируются до установки катушки: заранее спланируйте выезд из реакторного блока.
- Магнит включайте, когда робот уже стоит у объекта: захваченный на ходу стержень робот толкает в упор поста.
- **У консоли всего три попытки:** третий неверный `configure_charging(...)` перегревает её (`console-overheated`), и
  реактор в этом заезде не запустится. Проверяйте решение на сохранённой записи тестового заряда до отправки.
- Сборка топливной ячейки идёт 90 с: пока сборщик работает, займитесь другими задачами. Не трогайте топливную ячейку
  до конца сборки, а загруженные ячейки — пока топливная ячейка не установлена в реактор.
- После запуска реактора верните робота целиком в зону старта и финиша — заезд завершится успехом.

Зачётную попытку запускайте кнопкой «Новый прогон» (`F5`). `client.reset()` и `client.reset(seed)` после начала попытки
допустимы, но фиксируются в журнале и в «Итоге» (число сбросов и seed): жюри это видит.

In [ ]:
# Заезд

# 7.2.1 Техническая остановка реактора
# к первому шлагбауму
move_path([(2,4), (2, 6), (3,6)]);
wait_for_gate('gate.1');
# к реактору, нажимается кнопка
client.move_to(ID, 1.05, 2.85, speed=0.3);
# client.move_to(ID, 1.05, 2.85);
# и сразу на выход, ко второму шлагбауму
client.move_to(ID, 2.2, 2.6, speed=0.3);
wait_for_gate('gate.2');

# выход из зоны реактора
move_to(8, 7);


# 7.2.2 Подключение резервного питания
#sleep(3);
spool_x = None;
spool_y = None;
# очень странный формат вывода, не знаю, почему бы не сделать
# просто, например, objects как словарь, где ключом будет
# напр. spool или cell.2 или что-нибудь ещё, вместо этого приходится заниматься
for o in st()['objects']:
    if o['id'] == 'spool':
        spool_x = o['x']
        spool_y = o['y']
        break; # этим
# подъезжаем к катушке
# 0.259 - длина робота (берём с запасом)
client.move_to(ID, spool_x - SPOOL_START_OFFSET_X, spool_y); 
client.set_magnet(ID, True);
# приближаемся к катушке
client.set_velocity(ID, 0.07);
# пока не схватим катушкинс
client.wait_until(lambda s: s['robots'][0]['holding'] != None);
# опять отъезжаем обратно
client.move_to(ID, spool_x - SPOOL_START_OFFSET_X, spool_y);

move_to(8, 7);
# к приёмнику в зоне перезарядки
client.move_to(ID, 2.96, .3);
# отпустить катушку
client.set_magnet(ID, False);

charging_started_ms = st()['simulationTimeMs'];
# насчёт порядка:
# в общем, можно в любом порядке
# но, наверное, лучше сначала ячейки
# (фактически тот же код, что и для катушки)
# потом консоль, чтобы ускорить зарядку
# потом уже стержни
# (если что, можно потом будет поменять местами этап консоли и этап
# сборщика поменять местами, посмотреть, какой вариант быстрее)

# 7.2.3 Загрузка топливной ячейки
client.move_to(ID, 3, 1.4, speed=0.3);

move_to(6,4,rot=pi/2);
# данные о ячейках
# (главные поля: x, y, carrier)
he = [o for o in st()['objects'] if o['id'] in ['cell.1','cell.2']]

# чисто для удобства
# кто знает, какие данные будут там на полигоне, на порядок в массиве полагаться не стоит
# (наверное…)
if he[0]['x'] < he[1]['x']:
    he[0], he[1] = he[1], he[0];
# теперь в he[0] ячейка справа, а в he[1] ячейка слева
client.move_to(ID, he[0]['x'], 1.55);
# дальше всё как с катушкой: приближаемся, берём и едем
client.set_magnet(ID, True);
client.set_velocity(ID, 0.15);
client.set_magnet(ID, True); ##################
client.wait_until(lambda s: s['robots'][0]['holding'] != None);
# взяли ячейку (что здесь происходит?)
client.set_magnet(ID, True); ##################
client.stop(ID);
sleep(0.1);
client.set_magnet(ID, True); ##################
client.set_velocity(ID, -0.2);
sleep(0.1);
client.set_magnet(ID, True); ##################
move_to(2,4, y_offset=-0.05);
# к приёмнику и отпустить её
client.set_magnet(ID, True); ##################
client.move_to(ID, 0.8, .5, rotation=-pi/2);
client.set_magnet(ID, False);
# и обратно ко второй ячейке
move_to(2,4);
client.move_to(ID, he[1]['x'], 1.45, rotation=pi/2);

client.set_magnet(ID, True);
client.set_velocity(ID, 0.2);
client.wait_until(lambda s: s['robots'][0]['holding'] != None);
move_to(4,4, y_offset=0);
move_to(2,4, y_offset=-0.01);
# move_path([(4,4),(2,4)]);
client.move_to(ID, 0.7, 0.7, rotation=-pi/2);
client.move_to(ID, 0.9, 0.5);
# отпустили вторую ячейку
client.set_magnet(ID, False);
# здесь всё готово, неплохо было бы записать, когда именно начали:
assembly_started_ms = st()['simulationTimeMs'];

# и едем опять в зону реактора
move_to(2,3,y_offset=-0.2);
# повернуть робота вверх сразу (для стержней)
move_to(2,6,rot=pi/2);
move_to(3,6);
# move_path([ (2, 6), (3, 6)]);
wait_for_gate('gate.1');
move_to(3,7, x_offset=0.05, y_offset=0.18);
# сначала к консоли
client.set_velocity(ID, x=0, y=0.3);
client.wait_until(lambda s: 'console' in s['robots'][0]['zones']);
client.stop(ID);


# 7.2.5 Консоль сюда
# Здесь надо будет
task = client.get_console_task();
amogus = [
    o for o in task.keys()
] + [o.keys() for o in task['accumulators']];
#print(task);
client.journal(level='info',
               event_id='Task_4',
               message=str(amogus));


for i in range(3): # 3 попытки, не знаю, чем их отличить
    # здесь получается массив в 3 элемента - кандидат на
    # "среднюю" величину для каждого аккумулятора
    answers = [];
    for acc, j in zip(task['accumulators'], range(3)):
        times = acc['time'];
        currents = acc['current'];
        cmp = 1;
        if i == 0: # медианы
            asc = sorted(currents[:24]);
            cmp = (asc[11] + asc[12]) / 2;
        elif i == 1: # средние
            cmp = sum(currents[:24]) / 24;
        else: # усечённые средние?
            asc = sorted(currents)[4:-4];
            cmp = sum(asc) / len(asc);
        switch = next(t for t, ic in zip(times, currents)
            if ic < 0.97 * cmp);
        answers.append({
            "current_amperes": cmp,
            "switch_seconds": switch - task["sampleSeconds"] * .5});
    print(f"\nОтветы на попытку № {i+1}:");
    #pp(answers);
    answer = client.configure_charging(answers);
    if answer['accepted']: # yay!!!
        accelerated_charging_ms = st()['simulationTimeMs'];
        break;
    # not yay
    sleep(0.09);


# к стержням
# client.move_to(ID, 1.2, 2.6);
client.move_to(ID, THIRD_ROD_X, ROD_START_Y, speed=0.25);



# 7.2.4 Настройка регуляторов энергии
# к стержням
for i in range(2, -1, -1):
    # он сразу будет подъезжать к следующим стержням по диагонали, в принципе не так страшно
    client.move_to(ID, THIRD_ROD_X - ROD_DX * (2 - i), ROD_START_Y);
    #client.move_to(ID, THIRD_ROD_X, 3.04);
    # начинаем приближаться к стержню
    client.set_velocity(ID, 0.25);
    client.set_magnet(ID, True);
    # до контакта
    client.wait_until(lambda s: s['robots'][0]['holding'] != None);
    #pp(st()['robots'][0]); # просто чтоб увидеть Y-координату
    # вытягиваем
    client.set_velocity(ID, -0.12);
    client.wait_until(lambda s: s['rods'][i]['state'] == 'configured',interval = 0.08);
    # вытянули
    client.set_magnet(ID, False);
client.set_velocity(ID, x=-0.3, y=0.3);
sleep(0.3); # просто чтоб выйти из ячейки, в которой мы уже достаточно давно стоим

# 7.2.3. Загрузка топливной ячейки
move_to(3, 7, y_offset=0.1);
wait_for_gate('gate.1');
# к топливной ячейке
move_path([(3,6),(2,6)]);
move_to(2,2,rot=-pi/2);
cell_x, cell_y = next((o['x'],o['y']) for o in st()["objects"] if o['id'] == 'cell.assembler');
client.move_to(ID, cell_x, cell_y + SPOOL_START_OFFSET_X);
asm_remaining = st()['assembler']['remainingMs'] / 1000;
client.wait_until(lambda s: s['assembler']['state'] == 'ready', timeout=asm_remaining+1);
client.set_magnet(ID, True);
client.set_velocity(ID, 0.1);
client.wait_until(lambda s: s['robots'][0]['holding'] != None);
client.move_to(ID, 0.6, 0.8, speed=0.3);
move_to(2,6,rot=pi/2);
move_to(3,6);
# проехать в реакторную зону
wait_for_gate('gate.1');
# и где-то здесь столкновение было?
move_to(3,7,x_offset=0.05, y_offset=0.1);
move_to(2,8);
move_to(2,8,y_offset=0.1);
client.set_magnet(ID, False);

# 7.2.6. Запуск реактора
move_to(3, 7, y_offset=0.1);
wait_for_gate('gate.1');
move_path([(3,6), (2,6), (2,2), (6,2)]);
client.wait_until(lambda s: s["charging"]["percent"] >= 100, timeout=336);
for accumulator_id in ("accumulator.1", "accumulator.2", "accumulator.3"):
    client.activate_accumulator(accumulator_id);
    sleep(0.03);
move_path([(2,2),(2,4),(1,4)]); # finally!

In [ ]:
sleep(1.5); #
client.close();

### Замечания по ходу работы кода
Всё, что обнаруживаю до окончания заезда, чтоб не забыть потом и не останавливать
- Со шлагбаумами: сделал ожидание по state, просто на всякий случай

Вообще про шлагбаумы (позиционирование) из документации:
gate.1 (в южном заборе блока): основание стойки — с западной стороны проезда, внутри блока, к северу от стрелы. Проезжайте примерно на 3 см восточнее центра подхода. После проезда на север не поворачивайте на запад, пока корпус робота целиком не выйдет севернее основания: его северный край — примерно в 11 см севернее стрелы, а полуширина робота — около 10,5 см.
gate.2 (в восточном заборе блока): основание стойки — с северной стороны проезда, снаружи блока. Проезжайте примерно на 4 см южнее центра подхода. Южный забор блока продолжается на восток за стрелу ещё примерно на 27 см, а восточный забор начинается севернее стойки: перед шлагбаумом со стороны востока — «карман» между ними. Подъезжайте к нему с востока прямо, а не вдоль забора с юга.
